In [1]:
from LanguageDatasets import LanguageDataset
from dotenv import load_dotenv
import os
from datasets import load_dataset
import pandas as pd

def mergeFreelingDictionary(folder_path, output_file):
    """
    Combina todos los archivos .src del diccionario de FreeLing
    en un único archivo con una palabra por línea.
    Optimizado para velocidad.
    """
    dictionary_path = os.path.join(folder_path, "dictionary")
    lexicon = set()

    for filename in os.listdir(dictionary_path):

        full_path = os.path.join(dictionary_path, filename)

        with open(full_path, "r", encoding="utf8", errors="ignore") as f:
            for line in f:
                if not line or line.startswith("#"):
                    continue
                word = line.split(" ", 1)[0].lower() 
                lexicon.add(word)

    with open(output_file, "w", encoding="utf8") as out:
        out.write("\n".join(sorted(lexicon)))

    print(f"Lexicón generado: {output_file} ({len(lexicon)} palabras)")


## Gallego

In [32]:
import numpy as np
with open("datasets/gallego/idioms_train_es.txt", "r") as fEsp:
    esp = fEsp.readlines()
with open("datasets/gallego/idioms_train_gl.txt", "r") as fGl:
    gl = fGl.readlines()  
with open("datasets/gallego/idioms_test_es.txt", "r") as fEsp:
    espTest = fEsp.readlines()
with open("datasets/gallego/idioms_test_gl.txt", "r") as fGl:
    glTest = fGl.readlines()  

gallego = pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])
gallego.head()

,es,gl
0,Aquel pronto se convirtió en un movimiento de ...,Aquela arroutada converteuse nun movemento de ...
1,"La realidad, por dura que sea, siempre nos rec...","A realidade, por dura que sexa, sempre nos lem..."
2,"Mi abuela la palmó ayer, pero nos dejó llenos ...","A miña avoa espichou onte, mais deixounos cheo..."
3,El cantante aclaró la garganta haciendo amago ...,O cantante limpou a gorxa facendo ademán de co...
4,"El hombre, al ver a su exmujer en el bar, pagó...","O home, ao ver a súa exmuller no bar, pagou a ..."


## Aranés

In [35]:
# Login using e.g. `huggingface-cli login` to access this dataset
aranes = pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet")
aranes.head()

,es,arn
0,suprime la actividad enzimática?,Elègi de l’ensenhament actiu?
1,Aparecer de entre los muebles.,Ven d’entre dos muebles.
2,España es el lugar donde se decidirá,L’Espanha que va estar lo lòc on
3,La clave está en el retoque.,La clau es au trauc.
4,Y se pregunto: ¿Que será de Dios sin el Diablo?,Que se Pau capita pas de que ne serà del demai?


## Asturiano

In [30]:
# Login using e.g. `huggingface-cli login` to access this dataset
asturiano = pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet")
asturiano.head()

,es,ast
0,Faltan pocos días para el comienzo de las clas...,Falten pocos díes pal empiezu de les clases ne...
1,"Finalizar la planta nuclear Atucha II, que fue...",Nel 2005 creóse'l Programa de sofitu al desenv...
2,"Hipólito llevó su embajada, y ella fue allí a ...","Hipólito llevó la so embaxada, y ella foi allá..."
3,"A continuación intervino Pau Morales –ERC-, qu...",La cara ye trasunto d'una figura arcaica grieg...
4,El fragmento tiene conexiones con el Regimient...,El fragmentu tien conexones col Regiment de la...


In [36]:
print(gallego.shape)
print(aranes.shape)
print(asturiano.shape)

(6599, 2)
(419908, 2)
(704378, 2)


### 1. Traducir un Q&A a los idiomas

### 3. Corrección gramatical mediante introducción de errores

Para evaluar la capacidad de corrección, se parte de textos correctos en asturiano o aranés (por ejemplo, de Wikipedia). Un LLM grande introduce errores controlados de ortografía, morfología o sintaxis. El modelo evaluado debe corregirlos. La comparación con el texto original permite medir la calidad de la corrección.

- ```"llama-3.1-8b-instant"``` Mucho más rápido, pruebas
- ```"openai/gpt-oss-120b"``` Para dataset final
- ```"qwen/qwen3-32b"``` Probar

In [2]:
from generateDataset import generateDatasetOrtograficoAnotado, generateDatasetHuecos
load_dotenv("secrets.env")
ast = LanguageDataset("asturiano",True)
ast.json = ast[:20]

Descargando tatoeba para asturiano:
Completado con éxito


In [3]:
evalDataset = generateDatasetOrtograficoAnotado(ast, os.getenv("GROQ_API_KEY"), save=False, model="openai/gpt-oss-120b")
print(evalDataset.shape)

Progreso: 100.0% (20/20) | step: 3 s, remaining time: 0 min 0 ss
Dataset Generado
(20, 3)


In [4]:
for i in range(evalDataset.shape[0]):
    print(evalDataset["annotated"][i])
    print(evalDataset["original"][i])
    print(evalDataset["n_errors"][i])
    print("="*20,"\n")

Tas buenu <err t=ort>pá</err> dir a nengún sitiu.
Tas buenu pa dir a nengún sitiu.
1

El <err t=ort>cielú</err> del atapecer <err t=reord>roxu ye</err>.
El cielu del atapecer ye roxu.
2

Nun <err t=ort>gastez</err> más perres de les que ganes.
Nun gastes más perres de les que ganes.
1

Le<err t=ort>va</err> las llaves <err t=lex>para</err> <err t=drop>tu </err>hermano.
Lleva-y les llaves al to hermanu.
3

Esti <err t=ort>xergon</err> ye <err t=lex>vieja</err> y <err t=add>muy</err> máncame <err t=drop></err> renaz.
Esti xergón ye vieyu y máncame nel renaz.
4

<err t=ort>Fái</err> el favor de vestite, que vamos llegar tarde!
Fai el favor de vestite, que vamos llegar tarde!
1

Esa película ye afayadiza pa <err t=ort>xénte</err> de toles edaes.
Esa película ye afayadiza pa xente de toles edaes.
1

El pan se hace con <err t=ort>harinna</err>, <err t=drop>agua</err> y levadura.
El pan faise con farina, agua y formientu.
2

<err t=ort>Maestru</err>, <err t=drop>al </err><err t=ort>nun</err> 

In [ ]:
huecosDataset = generateDatasetHuecos(ast, False)
print(huecosDataset.shape)

(20, 3)


In [6]:
from huggingface_hub import login
load_dotenv("secrets.env")
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
from metricas.vocabulario import evaluacionHuecos
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Nombre del modelo en Hugging Face
model_name = "google/gemma-3-1b-it"

# 1. Cargar tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Cargar modelo
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto"   # opcional: usa GPU si está disponible
)
acc_vals = []
acc_low_vals = []
lev_vals = []
topk_acc_vals = []
for _, row in huecosDataset.iterrows():
    res = evaluacionHuecos(
        model, tokenizer,
        row["masked_sentence"],
        row["missing_word"],
        lang="ast"
    )
    acc_vals.append(res["accuracy"])
    acc_low_vals.append(res["accuracy_lower"])
    lev_vals.append(res["levenshtein"])
    topk_acc_vals.append(res["topk_accuracy"])
    print(res)
print(float(np.mean(acc_vals)))
print(float(np.mean(acc_low_vals)))
print(float(np.mean(lev_vals)))
print(float(np.mean(topk_acc_vals)))

2026-01-27 12:44:21.996892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'masked': '<mask> buenu pa dir a nengún sitiu.', 'missing_word': 'Tas', 'predicted': '', 'accuracy': 0.0, 'accuracy_lower': 0.0, 'levenshtein': 3, 'topk_accuracy': 0.0, 'topk_candidates': ['', '**buenu**']}
{'masked': 'El cielu del atapecer ye <mask>', 'missing_word': 'roxu.', 'predicted': 'El', 'accuracy': 0.0, 'accuracy_lower': 0.0, 'levenshtein': 5, 'topk_accuracy': 0.0, 'topk_candidates': ['El']}
{'masked': 'Nun gastes más perres de les <mask> ganes.', 'missing_word': 'que', 'predicted': '**copes**', 'accuracy': 0.0, 'accuracy_lower': 0.0, 'levenshtein': 8, 'topk_accuracy': 0.0, 'topk_candidates': ['**copes**', '**caja**', '**canchas**']}
{'masked': '<mask> les llaves al to hermanu.', 'missing_word': 'Lleva-y', 'predicted': '', 'accuracy': 0.0, 'accuracy_lower': 0.0, 'levenshtein': 7, 'topk_accuracy': 0.0, 'topk_candidates': ['']}
{'masked': 'Esti xergón ye <mask> y máncame nel renaz.', 'missing_word': 'vieyu', 'predicted': '```', 'accuracy': 0.0, 'accuracy_lower': 0.0, 'levenshte

NameError: name 'np' is not defined